# Quantization From Scratch

$$
\textbf{Range Selection}
$$

$$
\alpha = \max(x), \qquad \beta = \min(x)
$$

$$
\text{or (percentile range)}
$$

$$
\alpha = P_p(x), \qquad \beta = P_{1-p}(x)
$$

$$
\textbf{Asymmetric Quantization}
$$

$$
s = \frac{\alpha - \beta}{q_{\max} - q_{\min}}, 
\qquad
z = \left\lfloor q_{\min} - \frac{\beta}{s} \right\rceil
$$

$$
q = \mathrm{clip}
\left(
\left\lfloor \frac{x}{s} + z \right\rceil,
q_{\min},
q_{\max}
\right)
$$

$$
\hat{x} = s (q - z)
$$

$$
\textbf{Symmetric Quantization}
$$

$$
\alpha = \max(|x|)
$$

$$
s = \frac{\alpha}{2^{b-1}-1}
$$

$$
q = \mathrm{clip}
\left(
\left\lfloor \frac{x}{s} \right\rceil,
-(2^{b-1}-1),
2^{b-1}-1
\right)
$$

$$
\hat{x} = s q
$$

$$
\textbf{Quantization Ranges}
$$

$$
q_{\min} = 0, \qquad q_{\max} = 2^b - 1
$$

$$
q_{\min} = -(2^{b-1}-1), \qquad q_{\max} = 2^{b-1}-1
$$

In [1]:
import numpy as  np
from typing import Tuple

In [2]:
# Quantization helpers
def clamp(tensor: np.ndarray, lower_bound: int, upper_bound: int) -> np.ndarray:
    min_mask = tensor < lower_bound
    max_mask = tensor > upper_bound
    tensor[min_mask] = lower_bound
    tensor[max_mask] = upper_bound
    return tensor

def asymmetric_quantize(tensor: np.ndarray, bits: int) -> Tuple[np.ndarray, float, int]:
    alpha = np.max(tensor)
    beta = np.min(tensor)
    
    qmin = 0
    qmax = 2 ** bits - 1
    
    scale = (alpha - beta) / (qmax - qmin)
    z = int(np.round(-1 * beta / scale))
    quantized_tensor = np.round(tensor / scale + z)
    quantized_tensor = clamp(quantized_tensor, qmin, qmax).astype(np.int32)
    return quantized_tensor, scale, z

def asymmetric_dequantize(tensor: np.array, scale: float, z: int) -> np.array:
    return (tensor - z) * scale

def symmetric_quantize(tensor: np.array, bits: int) -> Tuple[np.array, float]:
    alpha = np.max(np.abs(tensor))

    qmax = 2 ** (bits - 1) - 1
    qmin = -qmax
    
    scale = alpha / qmax
    quantized_tensor = np.round(tensor / scale)
    quantized_tensor = clamp(quantized_tensor, qmin, qmax).astype(np.int32)
    return quantized_tensor, scale

def symmetric_dequantize(tensor: np.array, scale: float) -> np.array:
    return tensor * scale

def quantization_error(tensor: np.array, quantized_tensor: np.array) -> float:
    # Calculate MSE
    return np.mean((tensor - quantized_tensor) ** 2)

In [3]:
## Create tensor w/ random numbers

# Supress scientific notation
np.set_printoptions(suppress=True)

# Generate uniformly distributed parameters
params = np.random.uniform(low=-75, high=150, size=10)

# Set max and min values and 0 for debugging
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

# Round params
params = np.round(params, 2)
print("Original:\n", params, end="\n\n")

# Quantize the params
asq_params, asq_scale, asq_z = asymmetric_quantize(params, 8)
print("Asymmetric Quantization:\n", asq_params)
print(f"Asymmetric scale: {asq_scale}, z: {asq_z}", end="\n\n")

sq_params, sq_scale = symmetric_quantize(params, 8)
print("Symmetric Quantization:\n", sq_params)
print(f"Symmetric scale: {sq_scale}", end="\n\n")

Original:
 [145.95 -28.23   0.   -14.71 114.98  37.44  64.6  134.62 -27.23 119.72]

Asymmetric Quantization:
 [255   0  41  19 209  96 136 238   1 216]
Asymmetric scale: 0.6830588235294117, z: 41

Symmetric Quantization:
 [127 -25   0 -13 100  33  56 117 -24 104]
Symmetric scale: 1.1492125984251969



In [4]:
# Dequantize params back to 32 bits
asd_params = asymmetric_dequantize(asq_params, asq_scale, asq_z)
sd_params = symmetric_dequantize(sq_params, sq_scale)

print("Original:\n", params, end="\n\n")
print("Asymmetric Dequantization:\n", asd_params, end="\n\n")
print("Symmetric Dequantization:\n", sd_params, end="\n\n")

# Compute quantization error
asq_error = np.round(quantization_error(params, asd_params), 5)
sq_error = np.round(quantization_error(params, sd_params), 5)

print(f"Asymmetric Quantization error: {asq_error}")
print(f"Symmetric Quantization error: {sq_error}")

# Weights: Symmetric, Activations: Asymmetric, Extremely constrained hardware: Fully symmetric

Original:
 [145.95 -28.23   0.   -14.71 114.98  37.44  64.6  134.62 -27.23 119.72]

Asymmetric Dequantization:
 [146.17458824 -28.00541176   0.         -15.02729412 114.75388235
  37.56823529  64.89058824 134.56258824 -27.32235294 119.53529412]

Symmetric Dequantization:
 [145.95       -28.73031496   0.         -14.93976378 114.92125984
  37.92401575  64.35590551 134.45787402 -27.58110236 119.51811024]

Asymmetric Quantization error: 0.03995
Symmetric Quantization error: 0.07907


# Comparison of min-max and percentile range selection strategies

In [5]:
# Helper
def percentile_asymmetric_quantize(tensor: np.ndarray, bits: int, percentile: float) -> Tuple[np.ndarray, float, int]:
    alpha = np.percentile(tensor, percentile)
    beta = np.percentile(tensor, 100 - percentile)
    
    qmin = 0
    qmax = 2 ** bits - 1
    
    scale = (alpha - beta) / (qmax - qmin)
    z = int(np.round(-1 * beta / scale))
    quantized_tensor = np.round(tensor / scale + z)
    quantized_tensor = clamp(quantized_tensor, qmin, qmax).astype(np.int32)
    return quantized_tensor, scale, z

In [6]:
# Generate uniformly distributed parameters
params = np.random.uniform(low=-75, high=150, size=10000)

# Insert outlier
params[0] = 1000

# Round params
params = np.round(params, 2)
print("Original (1st 5 values):\n", params[:5], end="\n\n")

# Quantize the params using min-max
asq_params, asq_scale, asq_z = asymmetric_quantize(params, 8)
print("Asymmetric Quantization (min-max):\n", asq_params[:5])
print(f"Asymmetric scale: {asq_scale}, z: {asq_z}", end="\n\n")

# Quantize the params using percentile
asqp_params, asqp_scale, asqp_z = percentile_asymmetric_quantize(params, 8, 99.99)
print("Asymmetric Quantization (percentile):\n", asq_params[:5])
print(f"Asymmetric scale: {asq_scale}, z: {asq_z}", end="\n\n")

Original (1st 5 values):
 [1000.    -40.08  -33.11   30.81    6.57]

Asymmetric Quantization (min-max):
 [255   8  10  25  20]
Asymmetric scale: 4.215686274509804, z: 18

Asymmetric Quantization (percentile):
 [255   8  10  25  20]
Asymmetric scale: 4.215686274509804, z: 18



In [7]:
# Dequantize params back to 32 bits
asd_params = asymmetric_dequantize(asq_params, asq_scale, asq_z)
asdp_params = asymmetric_dequantize(asqp_params, asqp_scale, asqp_z)


print("Original:\n", params[:5], end="\n\n")
print("Asymmetric Dequantization (min-max):\n", asd_params[:5], end="\n\n")
print("Symmetric Dequantization (percentile):\n", asdp_params[:5], end="\n\n")

# # Compute quantization error
asq_error = np.round(quantization_error(params[1:], asd_params[1:]), 2)
asqp_error = np.round(quantization_error(params[1:], asdp_params[1:]), 2)

print(f"Asymmetric Quantization error (min-max): {asq_error}")
print(f"Asymmetric Quantization error (percentile): {asqp_error}")

Original:
 [1000.    -40.08  -33.11   30.81    6.57]

Asymmetric Dequantization (min-max):
 [999.11764706 -42.15686275 -33.7254902   29.50980392   8.43137255]

Symmetric Dequantization (percentile):
 [150.04333467 -39.71735329 -33.53909834  30.89127478   6.17825496]

Asymmetric Quantization error (min-max): 1.47
Asymmetric Quantization error (percentile): 0.07


# Post-training Quantization
- Quantizing a pre-trained model

In [8]:
import os

import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

In [9]:
# Set random seed
_ = torch.manual_seed(42)

In [10]:
# Helpers
def count_trainable_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

class DatasetLoader:
    def __init__(self, dataset_name='MNIST', batch_size=32):
        self.dataset_name = dataset_name
        self.batch_size = batch_size
        self.transform = transforms.ToTensor()

    def get_dataloader(self):
        dataset = datasets.MNIST(root="./data", train=True, download=True, transform=self.transform)
        val_size = int(0.1 * len(dataset))
        train_size = len(dataset) - val_size
        train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False)
        return train_loader, val_loader

def train(model, criterion, optimizer, data_loader, n_epochs=5):
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0
        for batch_idx, (data, target) in enumerate(data_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(data_loader)
        print(f"Epoch: {epoch}:: Average Loss: {avg_loss}")
    return

def test(model, data_loader):
    total = 0
    correct = 0
    model.eval()
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            for idx, i in enumerate(output):
                if torch.argmax(i) == target[idx]:
                    correct += 1
                total += 1

    print(f"Accuracy: {round(correct / total, 3)}")
    return

def print_model_size(model):
    torch.save(model.state_dict(), "tmp.pt")
    print(f"Model size (KB): {os.path.getsize('./tmp.pt') / 1e3}")
    os.remove("./tmp.pt")

In [11]:
# Constants
device = "cpu"

In [12]:
# Create simple MLP network
class SimpleNet(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        self.linear1 = nn.Linear(28 * 28, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x

In [13]:
# Train model
model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader, val_loader = DatasetLoader(dataset_name='MNIST', batch_size=64).get_dataloader()

train(model, criterion, optimizer, train_loader, n_epochs=5)

# Save model
MODEL_PATH = "./simple_net.pt"
if os.path.exists(MODEL_PATH ):
    model.load_state_dict(torch.load(MODEL_PATH ))
    print("Loaded SimpleNet")
else:
    torch.save(model.state_dict(), MODEL_PATH )

Epoch: 0:: Average Loss: 0.3435347944771721
Epoch: 1:: Average Loss: 0.13990052022763763
Epoch: 2:: Average Loss: 0.09573792483794429
Epoch: 3:: Average Loss: 0.07206302293315883
Epoch: 4:: Average Loss: 0.05410761101600621
Loaded SimpleNet


In [14]:
# Check layer-1 weights and size of model before quantization
print("Model Linear Layer-1 Weights before quantization:")
print(model.linear1.weight)
print(model.linear1.weight.dtype)

print("Model size before quantization:")
print_model_size(model)

print("Model Accuracy before quantization:")
test(model, val_loader)

Model Linear Layer-1 Weights before quantization:
Parameter containing:
tensor([[-0.0245, -0.0290, -0.0224,  ..., -0.0243,  0.0101, -0.0286],
        [ 0.0307, -0.0197, -0.0295,  ..., -0.0158, -0.0281,  0.0289],
        [-0.0215,  0.0214,  0.0251,  ...,  0.0131, -0.0257,  0.0084],
        ...,
        [-0.0347, -0.0293, -0.0055,  ...,  0.0190, -0.0285, -0.0150],
        [ 0.0201, -0.0081, -0.0005,  ..., -0.0222,  0.0239, -0.0194],
        [ 0.0075,  0.0317,  0.0059,  ..., -0.0302, -0.0052,  0.0079]],
       requires_grad=True)
torch.float32
Model size before quantization:
Model size (KB): 474.967
Model Accuracy before quantization:
Accuracy: 0.993


In [15]:
# Insert Min-Max Observers in copy of the trained model
class QuantizedSimpleNet(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        # Observers for quantization
        self.quantize = torch.quantization.QuantStub()
        self.linear1 = nn.Linear(28 * 28, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, 10)
        self.relu = nn.ReLU()
        # Observers for dequantization
        self.dequantize = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.quantize(x)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        x = self.dequantize(x)
        return x

In [16]:
quantized_model = QuantizedSimpleNet().to(device)

# Load weights from unquantized model and set to eval mode
quantized_model.load_state_dict(model.state_dict())
quantized_model.eval()

torch.backends.quantized.engine = "fbgemm" # x86 CPU

# Assign quantization configuration to submodules in `.qconfig` attribute
quantized_model.qconfig = torch.ao.quantization.get_default_qconfig(
    torch.backends.quantized.engine
)

# Prepare a copy of the model for quantization calibration or quantization-aware training
quantized_model = torch.ao.quantization.prepare(quantized_model, inplace=False) # Insert observers
quantized_model

/usr/local/lib/python3.8/dist-packages/torch/ao/quantization/observer.py:177: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QuantizedSimpleNet(
  (quantize): QuantStub(
    (activation_post_process): HistogramObserver()
  )
  (linear1): Linear(
    in_features=784, out_features=128, bias=True
    (activation_post_process): HistogramObserver()
  )
  (linear2): Linear(
    in_features=128, out_features=128, bias=True
    (activation_post_process): HistogramObserver()
  )
  (linear3): Linear(
    in_features=128, out_features=10, bias=True
    (activation_post_process): HistogramObserver()
  )
  (relu): ReLU()
  (dequantize): DeQuantStub()
)

In [17]:
# Calibrate the quantized model using test data
with torch.no_grad():
    for data, target in val_loader:
        data = data.to(device)
        quantized_model(data)

In [18]:
# Connvert to quantized modules
quantized_model = torch.ao.quantization.convert(quantized_model, inplace=False)

In [19]:
# Check statistics of various layers
quantized_model

QuantizedSimpleNet(
  (quantize): Quantize(scale=tensor([0.0079]), zero_point=tensor([0]), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=128, scale=0.2156502902507782, zero_point=77, qscheme=torch.per_channel_affine)
  (linear2): QuantizedLinear(in_features=128, out_features=128, scale=0.19353514909744263, zero_point=60, qscheme=torch.per_channel_affine)
  (linear3): QuantizedLinear(in_features=128, out_features=10, scale=0.6304082274436951, zero_point=76, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (dequantize): DeQuantize()
)

In [20]:
# Check layer-1 weights and size of model after quantization
print("Quantized Model Linear Layer-1 Weights after quantization:")
weight = quantized_model.linear1.weight()
print(weight)
print(weight.dtype)

print("Model size after quantization:")
print_model_size(quantized_model)

print("Model Accuracy after quantization:")
test(quantized_model, val_loader)

Quantized Model Linear Layer-1 Weights after quantization:
tensor([[-0.0253, -0.0281, -0.0225,  ..., -0.0253,  0.0112, -0.0281],
        [ 0.0320, -0.0192, -0.0288,  ..., -0.0160, -0.0288,  0.0288],
        [-0.0201,  0.0201,  0.0259,  ...,  0.0144, -0.0259,  0.0086],
        ...,
        [-0.0356, -0.0290, -0.0045,  ...,  0.0200, -0.0290, -0.0156],
        [ 0.0213, -0.0080,  0.0000,  ..., -0.0213,  0.0239, -0.0186],
        [ 0.0063,  0.0317,  0.0063,  ..., -0.0317, -0.0063,  0.0095]],
       size=(128, 784), dtype=torch.qint8,
       quantization_scheme=torch.per_channel_affine,
       scale=tensor([0.0028, 0.0032, 0.0029, 0.0022, 0.0030, 0.0033, 0.0025, 0.0045, 0.0030,
        0.0024, 0.0032, 0.0029, 0.0047, 0.0035, 0.0050, 0.0028, 0.0028, 0.0026,
        0.0032, 0.0036, 0.0031, 0.0028, 0.0026, 0.0042, 0.0028, 0.0023, 0.0044,
        0.0004, 0.0035, 0.0041, 0.0044, 0.0036, 0.0023, 0.0030, 0.0027, 0.0028,
        0.0027, 0.0030, 0.0028, 0.0022, 0.0035, 0.0026, 0.0029, 0.0027, 0.0026

# Quantization Aware Training

In [21]:
# Define model ready for Quantization Aware Training
class SimpleNet(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        # Observers for quantization
        self.quantize = torch.quantization.QuantStub()
        self.linear1 = nn.Linear(28 * 28, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, 10)
        self.relu = nn.ReLU()
        # Observers for dequantization
        self.dequantize = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.quantize(x)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        x = self.dequantize(x)
        return x

In [22]:
model = SimpleNet().to(device)
model.train()

torch.backends.quantized.engine = "fbgemm"   # x86 CPU

# QAT config
model.qconfig = torch.ao.quantization.get_default_qat_qconfig(
    torch.backends.quantized.engine
)

qat_model = torch.ao.quantization.prepare_qat(model, inplace=False)
qat_model # Model is not calibrated yet (no training/inference) thats why we have inf and -inf

SimpleNet(
  (quantize): QuantStub(
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (linear1): Linear(
    in_features=784, out_features=128, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
      (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([]), max_val=tensor([]))
    )
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor(

In [23]:
# Train the model
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(qat_model.parameters(), lr=1e-3)
train(qat_model, criterion, optimizer, train_loader)

Epoch: 0:: Average Loss: 0.4568743874844146
Epoch: 1:: Average Loss: 0.1220584859325599
Epoch: 2:: Average Loss: 0.08294170048839293
Epoch: 3:: Average Loss: 0.06137121071425483
Epoch: 4:: Average Loss: 0.04625230831131604


In [24]:
# Check statistics of various layers
qat_model

SimpleNet(
  (quantize): QuantStub(
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([0.0079]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=0.0, max_val=1.0)
    )
  )
  (linear1): Linear(
    in_features=784, out_features=128, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([0.0021, 0.0022, 0.0026, 0.0018, 0.0020, 0.0023, 0.0023, 0.0024, 0.0020,
              0.0024, 0.0021, 0.0019, 0.0019, 0.0019, 0.0019, 0.0020, 0.0018, 0.0022,
              0.0019, 0.0019, 0.0019, 0.0020, 0.0020, 0.0021, 0.0027, 0.0019, 0.0021,
              0.0024, 0.0022, 0.0020, 0.0021, 0.0024, 0.0022, 0.0019, 0.0019, 0.0022,
              0.0020, 0.0018, 0.0

In [25]:
# Quantize the model using the statistics collected during training
qat_model.eval()
quantized_model = torch.ao.quantization.convert(qat_model, inplace=False)

In [26]:
# Check statistics of various layers
quantized_model

SimpleNet(
  (quantize): Quantize(scale=tensor([0.0079]), zero_point=tensor([0]), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=128, scale=0.13003329932689667, zero_point=78, qscheme=torch.per_channel_affine)
  (linear2): QuantizedLinear(in_features=128, out_features=128, scale=0.12325309216976166, zero_point=56, qscheme=torch.per_channel_affine)
  (linear3): QuantizedLinear(in_features=128, out_features=10, scale=0.33846959471702576, zero_point=84, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (dequantize): DeQuantize()
)

In [27]:
print("Quantized Model Linear Layer-1 Weights after quantization:")
weight = quantized_model.linear1.weight()
print(torch.int_repr(weight))
print(weight.dtype)

print("Model size after quantization:")
print_model_size(quantized_model)

Quantized Model Linear Layer-1 Weights after quantization:
tensor([[  1,  10, -10,  ..., -16,  -2,  12],
        [  2, -12,   4,  ..., -15,  -9,   2],
        [ 11,  -8,   6,  ...,   3,  -7,  -9],
        ...,
        [ -8, -17,  -5,  ...,  19,   1,  -6],
        [ -1,  -9, -12,  ...,  -9,  12,  11],
        [ 12,  -9, -20,  ...,  -1,  20,  -4]], dtype=torch.int8)
torch.qint8
Model size after quantization:
Model size (KB): 129.193


In [28]:
# Test the model after QAT
print("Model Accuracy after quantization:")
test(quantized_model, val_loader)

Model Accuracy after quantization:
Accuracy: 0.974
